# TCGA-PAAD + AlphaMissense Integration Pipeline

This notebook builds a patient–mutation pair annotation table combining:
- Somatic mutations from TCGA-PAAD (The Cancer Genome Atlas - Pancreatic Adenocarcinoma)
- Pathogenicity scores from AlphaMissense (hg38)
- Molecular subtype / tumour purity from Bailey et al. (2016)

**Pipeline steps:**

1. Download the **TCGA-PAAD somatic mutation MAF** from cBioPortal
2. Filter to primary tumour missense mutations; rename all columns to `snake_case`
3. Add `subtype` and `purity` from Bailey et al. (2016) Table S1 (in-memory join on `patient_barcode`)
4. Download **AlphaMissense hg38**
5. Convert AlphaMissense TSV → Parquet
6. Join TCGA-PAAD × AlphaMissense — two outputs:
   - **Inner join** (`tcga_paad_alphamissense_inner.parquet`): only mutations with AM scores; `uniprot_id` always from AM
   - **Left join** (`tcga_paad_alphamissense_left.parquet`): all 16,086 missense mutations; `uniprot_id` from AM, null for unmatched rows

Both outputs share the same **14-column schema**:

| Column | Type | Source | Description |
|--------|------|--------|-------------|
| `chrom` | String | MAF | Chromosome with `chr` prefix (e.g. `chr1`) |
| `pos` | Int64 | MAF | Genomic position (GRCh38) |
| `ref` | String | MAF | Reference nucleotide |
| `alt` | String | MAF | Alternate nucleotide |
| `patient_barcode` | String | MAF | 12-character TCGA patient identifier (e.g. `TCGA-HZ-7922`) |
| `gene_symbol` | String | MAF | HGNC gene symbol (e.g. `KRAS`) |
| `uniprot_id` | String | AM | UniProt accession; null for rows not covered by AlphaMissense |
| `transcript_id` | String | AM | Ensembl transcript (e.g. `ENST00000256078`); null for left-join-only rows |
| `protein_variant` | String | AM | Amino acid substitution key (e.g. `G12D`); null for left-join-only rows |
| `am_pathogenicity` | Float64 | AM | Pathogenicity score 0–1; null for left-join-only rows |
| `am_class` | String | AM | `likely_benign` / `ambiguous` / `likely_pathogenic`; null for left-join-only rows |
| `af_1000g` | Float64 | MAF | 1000 Genomes allele frequency from VEP annotation; null if variant not in 1000G |
| `subtype` | String | Bailey et al. (2016) | Moffitt molecular subtype: `classical` or `basal-like`; null if unclassified |
| `purity` | String | Bailey et al. (2016) | Tumour purity tier: `high` or `low`; null if not in high-purity subset |

> **`_inner.parquet`** is the input to the TCGA + 1000G merge pipeline.  
> **`_left.parquet`** is used for standalone TCGA mutation landscape analysis.  
> The 391 rows exclusive to the left join are mutations in large proteins (>7,500 aa) absent from AlphaMissense (e.g. DST/Dystonin); their `uniprot_id` is null.

## Prerequisites

```bash
pip install polars matplotlib openpyxl
```

All downloaded and generated files are stored in `C:\Users\daane\Desktop\DATA\TCGA-PAAD`.  
Adjust `DATA_DIR` before running on a different machine.

Required files not downloaded by this notebook:
- `Table_S1_Integrated_Genomic_Characterization_of_Pancreatic_Ductal_Adenocarcinoma.xlsx` — Bailey et al. (2016) Table S1, required for Step 3

## Imports and Path Constants

All imports and file paths are defined here so every subsequent step can simply reference the constants. The data directory is created if it does not yet exist.

In [ ]:
import os
import gzip
import shutil
import tarfile
import urllib

import openpyxl
import polars as pl
import matplotlib.pyplot as plt

DATA_DIR = r"C:\Users\daane\Desktop\DATA\TCGA-PAAD"
os.makedirs(DATA_DIR, exist_ok=True)

# TCGA-PAAD — cBioPortal
ARCHIVE_URL  = "https://datahub.assets.cbioportal.org/paad_tcga_gdc.tar.gz"
ARCHIVE_PATH = os.path.join(DATA_DIR, "paad_tcga_gdc.tar.gz")
MAF_PATH     = os.path.join(DATA_DIR, "data_mutations.txt")

# AlphaMissense
AM_DOWNLOAD_LINK = "https://zenodo.org/records/10813168/files/AlphaMissense_hg38.tsv.gz?download=1"
AM_GZ_PATH       = os.path.join(DATA_DIR, "AlphaMissense_hg38.tsv.gz")
AM_TSV_PATH      = os.path.join(DATA_DIR, "AlphaMissense_hg38.tsv")
AM_PARQUET_PATH  = os.path.join(DATA_DIR, "alphamissense_hg38.parquet")

# Bailey et al. (2016) Table S1 for subtype classification (Step 3)
EXCEL_PATH = os.path.join(DATA_DIR, "Table_S1_Integrated_Genomic_Characterization_of_Pancreatic_Ductal_Adenocarcinoma.xlsx")

# Outputs — two AM joins, same schema
ANNOTATED_INNER_PATH = os.path.join(DATA_DIR, "tcga_paad_alphamissense_inner.parquet")
ANNOTATED_LEFT_PATH  = os.path.join(DATA_DIR, "tcga_paad_alphamissense_left.parquet")

# Schema constants
COORD_KEY = ["chrom", "pos", "ref", "alt"]

# All 14 output columns (same for both inner and left join outputs)
TARGET_COLS = [
    *COORD_KEY, "patient_barcode",
    "gene_symbol", "uniprot_id", "transcript_id",
    "protein_variant", "am_pathogenicity", "am_class",
    "af_1000g",
    "subtype", "purity",
]

DRIVER_GENES = ["KRAS", "TP53", "SMAD4", "CDKN2A"]

---
## Step 1 — Download TCGA-PAAD MAF

The TCGA-PAAD somatic mutation data is provided by cBioPortal as part of the `paad_tcga_gdc` study. The full study archive contains mutations, copy-number, clinical, and other data files; we extract only `data_mutations.txt`, the **masked somatic mutation MAF** (Mutation Annotation Format). This file is open-access — no dbGaP authorisation is required.

The MAF format is a tab-separated file with one row per somatic mutation per patient, annotated with genomic coordinates (GRCh38), variant classification, protein-level notation, and functional annotations from the Ensembl Variant Effect Predictor (VEP).

- **Study**: TCGA-PAAD GDC (`paad_tcga_gdc`)
- **Reference**: Raphael et al. (2017). *Integrated Genomic Characterization of Pancreatic Ductal Adenocarcinoma.* Cancer Cell, 32(2), 185–203. https://doi.org/10.1016/j.ccell.2017.07.007
- **Build**: GRCh38 | **Samples**: 186 total; 173 primary tumour samples (code `01`) from 168 unique patients

**Output**: `data_mutations.txt` (42.0 MB)

In [ ]:
OVERRIDE = False

if os.path.exists(MAF_PATH) and not OVERRIDE:
    print(f"Already exists: {MAF_PATH}, skipping download.")
else:
    print(f"Downloading {ARCHIVE_URL} ...")
    urllib.request.urlretrieve(ARCHIVE_URL, ARCHIVE_PATH)
    print(f"  {os.path.getsize(ARCHIVE_PATH) / 1e6:.1f} MB")

    print("Extracting data_mutations.txt ...")
    with tarfile.open(ARCHIVE_PATH) as tar:
        member = next(
            (m for m in tar.getmembers() if m.name.endswith("data_mutations.txt")),
            None,
        )
        if member is None:
            raise FileNotFoundError("data_mutations.txt not found in archive")
        with tar.extractfile(member) as f_in, open(MAF_PATH, "wb") as f_out:
            shutil.copyfileobj(f_in, f_out)

    os.remove(ARCHIVE_PATH)
    print(f"Deleted {ARCHIVE_PATH}")
    print("Done.")

---
## Step 2 — Load and Filter TCGA-PAAD

We load the full MAF file and apply two filters:

1. **Primary tumours only**: `Tumor_Sample_Barcode` encodes sample type at characters 14–15 (0-indexed). Code `01` = primary solid tumour; excludes the one metastasis sample (code `06`).
2. **Missense mutations only**: AlphaMissense scores amino acid substitutions exclusively. Keeping only `Missense_Mutation` rows ensures every variant has a potential AM match.

After filtering, we:
- Extract the 12-character **patient barcode** from `Tumor_Sample_Barcode` (characters 0–11; e.g. `TCGA-HZ-7922`)
- Add `chrom` with the `chr` prefix required by AlphaMissense (MAF stores bare chromosome numbers: `1` → `chr1`)
- Rename and standardise **all** column names to `snake_case`:

| MAF column | Renamed to | Notes |
|---|---|---|
| `Start_Position` | `pos` | genomic coordinate |
| `Reference_Allele` | `ref` | genomic coordinate |
| `Tumor_Seq_Allele2` | `alt` | genomic coordinate |
| `Hugo_Symbol` | `gene_symbol` | HGNC gene symbol |
| `1000G_AF` | `af_1000g` | 1000 Genomes allele frequency from VEP annotation |

- Select only the 7 target columns; all other per-sample sequencing metrics are discarded

**Output**: in-memory `tcga` DataFrame — 7 columns, 16,086 rows

| Column | Type | Description |
|--------|------|-------------|
| `chrom` | String | Chromosome (e.g. `chr1`) |
| `pos` | Int64 | Genomic position (GRCh38) |
| `ref` | String | Reference allele |
| `alt` | String | Alternate allele |
| `patient_barcode` | String | 12-character patient identifier |
| `gene_symbol` | String | HGNC gene symbol (e.g. `KRAS`) |
| `af_1000g` | Float64 | 1000 Genomes allele frequency; null if variant not in 1000G |

After filtering: **16,086 rows**, **168 unique patients**, **8,763 rows dropped** (non-primary samples + non-missense variants). `af_1000g` is null for 15,551 mutations (variants absent from 1000 Genomes VEP annotation).

In [ ]:
tcga_raw = pl.read_csv(
    MAF_PATH,
    separator="\t",
    comment_prefix="#",
    null_values=["", ".", "NA"],
    infer_schema_length=10_000,
)

tcga = (
    tcga_raw
    .filter(pl.col("Tumor_Sample_Barcode").str.slice(13, 2) == "01")
    .filter(pl.col("Variant_Classification") == "Missense_Mutation")
    .with_columns(
        pl.col("Tumor_Sample_Barcode").str.slice(0, 12).alias("patient_barcode"),
        ("chr" + pl.col("Chromosome")).alias("chrom"),
    )
    .rename({
        "Start_Position":    "pos",
        "Reference_Allele":  "ref",
        "Tumor_Seq_Allele2": "alt",
        "Hugo_Symbol":       "gene_symbol",
        "1000G_AF":          "af_1000g",
    })
    .with_columns(
        pl.col("pos").cast(pl.Int64),
    )
    .select([*COORD_KEY, "patient_barcode", "gene_symbol", "af_1000g"])
)

In [ ]:
# ── Inspection: Step 2 ────────────────────────────────────────
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
print(f"Rows in MAF (all variants, all types):  {tcga_raw.height:,}")
print(f"Rows after filter (primary, missense):  {tcga.height:,}")
print(f"Rows dropped:                           {tcga_raw.height - tcga.height:,}")
print(f"Unique patients:                        {tcga['patient_barcode'].n_unique():,}")
print(f"Columns ({tcga.width}): {tcga.columns}")
print()
print("Null counts per column:")
print(tcga.null_count())

---
## Step 3 — Add Subtype Classification

Adds `subtype` (Moffitt molecular subtype) and `purity` (tumour purity tier) to the in-memory `tcga` DataFrame via a left join on `patient_barcode`. Performing this before the AlphaMissense join ensures both `_inner` and `_left` outputs contain these columns without any additional read/write.

**Source**: Table S1 (worksheet 1) from:
> Bailey P. et al. (2016). *Genomic analyses identify molecular subtypes of pancreatic cancer.*  
> Nature, 531(7592), 47–52. https://doi.org/10.1038/nature16965

The worksheet has two header rows (rows 0–1 are skipped). Each subsequent row is one tumour sample.

**Column K (index 10) — Moffitt purity tier:**
- numeric value → `purity = "high"`
- `"LowPurity"` → `purity = "low"`
- `null` → `purity = null`

**Column L (index 11) — Moffitt subtype:**
- `1` → `subtype = "basal-like"`
- `2` → `subtype = "classical"`
- other/null → `subtype = null`

The Excel stores full tumour barcodes (e.g. `TCGA-IB-7651-01A`); we strip to the 12-character patient barcode (`TCGA-IB-7651`) to join on `patient_barcode`. Patients without a match receive `null` for both columns.

**Output**: 16,086 rows × 9 cols (no rows added or dropped by the left join).

After join: **classical** 13,744 mutations · **basal-like** 1,591 · **null** 751 (patients absent from Bailey Table S1). Purity: **high** 2,371 · **low** 12,964 · **null** 751.

In [ ]:
def parse_subtype_lookup(path):
    wb = openpyxl.load_workbook(path, read_only=True, data_only=True)
    ws = wb.worksheets[0]
    records = []
    for i, row in enumerate(ws.iter_rows(values_only=True)):
        if i < 2:           # row 0 = title, row 1 = header
            continue
        tumor_id = row[0]
        if tumor_id is None:
            continue
        patient_barcode = "-".join(str(tumor_id).split("-")[:3])
        purity_val  = row[10]   # column K: high-purity Moffitt cluster
        subtype_val = row[11]   # column L: all-sample Moffitt cluster
        purity  = "low" if purity_val == "LowPurity" else ("high" if purity_val is not None else None)
        subtype = {1: "basal-like", 2: "classical"}.get(subtype_val)
        records.append({"patient_barcode": patient_barcode, "subtype": subtype, "purity": purity})
    wb.close()
    return pl.DataFrame(records, schema={"patient_barcode": pl.Utf8, "subtype": pl.Utf8, "purity": pl.Utf8})

_tcga_cols_before = tcga.columns.copy()

lookup = parse_subtype_lookup(EXCEL_PATH)
tcga = tcga.join(lookup, on="patient_barcode", how="left")

In [ ]:
# ── Inspection: Step 3 ────────────────────────────────────────
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
print(f"Bailey Table S1: {len(lookup):,} samples")
print(f"Cols added: {set(tcga.columns) - set(_tcga_cols_before)}")
print(f"Shape: {tcga.shape}  ({tcga.shape[0]:,} rows × {tcga.shape[1]} cols)")
print()
print("Subtype value counts:")
print(tcga["subtype"].value_counts(sort=True))
print()
print("Purity value counts:")
print(tcga["purity"].value_counts(sort=True))
print()
print("Null counts — subtype / purity:")
print(tcga.select(["subtype", "purity"]).null_count())

---
## Step 4 — Download AlphaMissense hg38

**AlphaMissense** is a model developed by Google DeepMind that leverages AlphaFold protein structure predictions to score the pathogenicity of every possible single-nucleotide missense variant in the human proteome. Each variant receives:

- `am_pathogenicity`: a continuous score from **0** (benign) to **1** (pathogenic)
- `am_class`: a discrete label: `likely_benign`, `ambiguous`, or `likely_pathogenic`

We use the **hg38** build to match the GRCh38 reference genome used by TCGA-PAAD, 1000G, and gnomAD — enabling a unified join key across all three datasets.

**Coverage note**: AlphaMissense does not cover all proteins. Large proteins (>7,500 aa) such as DST/Dystonin are absent because AlphaFold2 cannot model them at full length. Approximately 194 genes have no AM scores; these variants will only appear in the left join output.

- **Source**: Zenodo, record 10813168
- **Reference**: Cheng et al. (2023). *Accurate proteome-wide missense variant effect prediction with AlphaMissense.* Science, 381(6664), eadg7492. https://doi.org/10.1126/science.adg7492

**Output**: `alphamissense_hg38.parquet` (the `.tsv.gz` is downloaded and decompressed temporarily; the TSV is deleted after conversion in Step 5)

In [ ]:
OVERRIDE = False

if os.path.exists(AM_PARQUET_PATH) and not OVERRIDE:
    print(f"Already exists: {AM_PARQUET_PATH}, skipping download.")
else:
    print(f"Downloading AlphaMissense to {AM_GZ_PATH} ...")
    urllib.request.urlretrieve(AM_DOWNLOAD_LINK, AM_GZ_PATH)
    print(f"  {os.path.getsize(AM_GZ_PATH) / 1e6:.1f} MB")

    print(f"Decompressing to {AM_TSV_PATH} ...")
    with gzip.open(AM_GZ_PATH, "rb") as f_in, open(AM_TSV_PATH, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

    os.remove(AM_GZ_PATH)
    print(f"Deleted {AM_GZ_PATH}")
    print("Done.")

---
## Step 5 — Convert AlphaMissense TSV → Parquet

The AlphaMissense TSV has no conventional header row — the file begins with `#`-prefixed comment lines followed by data rows without column names. We skip comments (`comment_prefix="#"`) and assign column names explicitly.

We drop the `genome` column (always `hg38`). The remaining columns propagated to the output are:

| Column | Description |
|---|---|
| `chrom` | Chromosome (e.g. `chr1`) |
| `pos` | GRCh38 genomic position |
| `ref` | Reference allele |
| `alt` | Alternate allele |
| `uniprot_id` | UniProt accession of the affected protein |
| `transcript_id` | Ensembl transcript identifier |
| `protein_variant` | Amino acid substitution (e.g. `G12D`) |
| `am_pathogenicity` | Pathogenicity score (0–1) |
| `am_class` | `likely_benign` / `ambiguous` / `likely_pathogenic` |

The conversion streams from disk to disk using Polars' lazy API so the full ~1.2 GB TSV is never loaded into memory.

**Input**: `AlphaMissense_hg38.tsv`  
**Output**: `alphamissense_hg38.parquet` — **71,697,556 rows**, 9 columns, 564 MB; no nulls in any column.

In [ ]:
OVERRIDE = False

if os.path.exists(AM_PARQUET_PATH) and not OVERRIDE:
    print(f"Already exists: {AM_PARQUET_PATH}, skipping.")
else:
    am_columns = [
        "chrom", "pos", "ref", "alt",
        "genome",
        "uniprot_id", "transcript_id", "protein_variant",
        "am_pathogenicity", "am_class",
    ]

    print(f"Streaming {AM_TSV_PATH} -> {AM_PARQUET_PATH} ...")
    (
        pl.scan_csv(
            AM_TSV_PATH,
            separator="\t",
            comment_prefix="#",
            has_header=False,
            new_columns=am_columns,
        )
        .drop("genome")
        .sink_parquet(AM_PARQUET_PATH)
    )

    os.remove(AM_TSV_PATH)
    print(f"Deleted {AM_TSV_PATH}")
    print("Done.")

---
## Step 6 — Join TCGA-PAAD × AlphaMissense

We annotate the mutation table by joining against AlphaMissense hg38 on genomic coordinates `[chrom, pos, ref, alt]`. Two output files are produced from a single AM scan.

**Deduplication**: AlphaMissense stores one row per (variant, transcript). Before joining, we reduce AM to one row per `(chrom, pos, ref, alt)` by keeping the transcript with the **highest `am_pathogenicity`** (ties broken by `transcript_id` ascending to ensure determinism).

---

### Inner join → `tcga_paad_alphamissense_inner.parquet`

Retains only mutations with an AlphaMissense match. `uniprot_id` is sourced entirely from AM.

| | Rows | Columns |
|---|---|---|
| Input: `tcga` (in memory) | 16,086 | 9 |
| Input: `alphamissense_hg38.parquet` | 71,697,556 | 9 |
| **Output** | **15,695** | **14** |

Excluded: 391 mutations in large proteins not covered by AlphaMissense (e.g. DST, ZFHX4, ASTN1). Null counts in output: `uniprot_id` 0 · `af_1000g` 15,171 (variants absent from 1000G VEP) · `subtype`/`purity` 726 (patients not in Bailey Table S1). No duplicate `(chrom, pos, ref, alt, patient_barcode)` keys.

**Used by**: the TCGA + 1000G merge pipeline.

---

### Left join → `tcga_paad_alphamissense_left.parquet`

Retains all 16,086 missense mutations. For rows without an AM match, `uniprot_id` and all AM score columns (`transcript_id`, `protein_variant`, `am_pathogenicity`, `am_class`) are null.

| | Rows | Columns |
|---|---|---|
| Input: `tcga` (in memory) | 16,086 | 9 |
| Input: `alphamissense_hg38.parquet` | 71,697,556 | 9 |
| **Output** | **16,086** | **14** |

The 391 rows exclusive to this output have null `uniprot_id`, `transcript_id`, `protein_variant`, `am_pathogenicity`, and `am_class`.

**Used by**: standalone TCGA mutation landscape analysis.

---

Both outputs contain the same 14 columns defined by `TARGET_COLS`:
`chrom`, `pos`, `ref`, `alt`, `patient_barcode`, `gene_symbol`, `uniprot_id`, `transcript_id`, `protein_variant`, `am_pathogenicity`, `am_class`, `af_1000g`, `subtype`, `purity`

In [ ]:
OVERRIDE = False

if os.path.exists(ANNOTATED_INNER_PATH) and os.path.exists(ANNOTATED_LEFT_PATH) and not OVERRIDE:
    print(f"Both outputs exist, skipping.")
    print(f"  Inner: {ANNOTATED_INNER_PATH}")
    print(f"  Left:  {ANNOTATED_LEFT_PATH}")
else:
    print(f"TCGA rows before join: {tcga.height:,}")

    am = (
        pl.scan_parquet(AM_PARQUET_PATH)
        .select([*COORD_KEY, "uniprot_id", "transcript_id", "protein_variant", "am_pathogenicity", "am_class"])
        .with_columns(pl.col("pos").cast(pl.Int64))
        # Keep the transcript with the highest am_pathogenicity per variant; tiebreak alphabetically
        .sort(["am_pathogenicity", "transcript_id"], descending=[True, False], nulls_last=True)
        .unique(subset=COORD_KEY, keep="first", maintain_order=True)
    )

    # --- Inner join: only AM-scored rows; uniprot_id always from AM ---
    inner = (
        tcga.lazy()
        .join(am, on=COORD_KEY, how="inner")
        .select(TARGET_COLS)
        .collect()
    )
    inner.write_parquet(ANNOTATED_INNER_PATH)
    print(f"Inner join written: {inner.height:,} rows × {inner.width} cols → {ANNOTATED_INNER_PATH}")

    # --- Left join: all missense rows; uniprot_id from AM, null for unmatched rows ---
    left = (
        tcga.lazy()
        .join(am, on=COORD_KEY, how="left")
        .select(TARGET_COLS)
        .collect()
    )
    left.write_parquet(ANNOTATED_LEFT_PATH)
    print(f"Left join written:  {left.height:,} rows × {left.width} cols → {ANNOTATED_LEFT_PATH}")

In [ ]:
key_cols = ["chrom", "pos", "ref", "alt", "patient_barcode"]

dupes = (
    inner
    .group_by(key_cols)
    .len()
    .filter(pl.col("len") > 1)
)

if len(dupes) == 0:
    print("No duplicate keys — every (chrom, pos, ref, alt, patient_barcode) is unique.")
else:
    print(f"WARNING: {len(dupes):,} duplicate key groups ({dupes['len'].sum():,} rows affected).\n")

    dupes_full = inner.join(dupes.select(key_cols), on=key_cols, how="inner")

    non_key_cols = [c for c in inner.columns if c not in key_cols]
    print("Variation within duplicate key groups (per column):")
    for col in non_key_cols:
        n_varying = (
            dupes_full
            .group_by(key_cols)
            .agg(pl.col(col).n_unique().alias("n_unique"))
            .filter(pl.col("n_unique") > 1)
            .height
        )
        label = f"varies in {n_varying:,} / {len(dupes):,} groups" if n_varying > 0 else "constant in all groups"
        print(f"  {col:30s}  {label}")

    print(f"\nTop duplicate groups (most copies):")
    print(dupes.sort("len", descending=True).head(10))

---
## Summary

| Step | Input | Output | Notes |
|------|-------|--------|-------|
| 1 — Download TCGA-PAAD MAF | cBioPortal URL | `data_mutations.txt` (42.0 MB) | Extracts from tar.gz; skips if file exists |
| 2 — Load and filter | `data_mutations.txt` | in-memory `tcga` (7 cols, 16,086 rows, 168 patients) | Primary tumours (code `01`); missense only; 8,763 rows dropped; all columns renamed to `snake_case` |
| 3 — Add subtype | `tcga` (7 cols) + `Table_S1_...xlsx` | `tcga` (9 cols) | Bailey et al. (2016) Moffitt subtypes; left join on `patient_barcode`; classical 13,744 · basal-like 1,591 · null 751 |
| 4 — Download AlphaMissense | Zenodo URL | `alphamissense_hg38.parquet` | ~1.2 GB compressed; TSV decompressed then deleted after conversion; skips if Parquet exists |
| 5 — Convert AlphaMissense | `AlphaMissense_hg38.tsv` | `alphamissense_hg38.parquet` (71,697,556 rows, 9 cols, 564 MB) | Streamed via lazy API; drops `genome` column |
| 6 — Inner join | `tcga` + `alphamissense_hg38.parquet` | `tcga_paad_alphamissense_inner.parquet` (14 cols, 15,695 rows) | Only AM-scored mutations; 391 large-protein mutations excluded; `uniprot_id` from AM; used by merge pipeline |
| 6 — Left join | `tcga` + `alphamissense_hg38.parquet` | `tcga_paad_alphamissense_left.parquet` (14 cols, 16,086 rows) | All missense mutations; 391 rows have null AM columns; used for standalone analysis |